In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Data Pipeline

In [ ]:
df = pd.read_csv("GALAXY_DATASET/final_15000.csv")
df.head()

In [ ]:
img_folder = "Images Differing sizes/224"

# 10 images are missing from the folder, drop those rows
valid = set(int(f.replace(".jpg", "")) for f in os.listdir(img_folder) if f.endswith(".jpg"))
df = df[df["asset_id"].isin(valid)].reset_index(drop=True)
print(len(df))  # should be 14990

In [ ]:
def make_label(row, thresh=0.5):

    smooth    = row["t01_smooth_or_features_a01_smooth_debiased"]
    featured  = row["t01_smooth_or_features_a02_features_or_disk_debiased"]
    irregular = row["t08_odd_feature_a22_irregular_debiased"]
    bar       = row["t03_bar_a06_bar_debiased"]
    spiral    = row["t04_spiral_a08_spiral_debiased"]
    edgeon    = row["t02_edgeon_a04_yes_debiased"]

    labels = []

    if featured > smooth:
        labels.append("featured")
    else:
        labels.append("smooth")

    if irregular > bar:
        labels.append("irregular")
        if edgeon > thresh:
            labels.append("edge_on")

    else:
        if bar <= thresh and spiral <= thresh and edgeon <= thresh:
            labels.append("elliptical")
        else:
            if bar > thresh:
                labels.append("barred_spiral")
            elif spiral > thresh:
                labels.append("spiral")
            if edgeon > thresh:
                labels.append("edge_on")

    return "_".join(labels)

df["label"] = df.apply(make_label, axis=1)
df = df.dropna(subset=["label"]).reset_index(drop=True)
print(df["label"].value_counts())

In [ ]:
classes = sorted(df["label"].dropna().unique())

print(classes)
print(f"Number of classes: {len(classes)}")

class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

df["label_idx"] = df["label"].map(class_to_idx)

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path = os.path.join(self.img_dir, f"{int(row['asset_id'])}.jpg")
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label_idx"])

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_loader = DataLoader(GalaxyDataset(train_df, img_folder, train_tfms), batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(GalaxyDataset(val_df,   img_folder, val_tfms),   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(GalaxyDataset(test_df,  img_folder, val_tfms),   batch_size=32, shuffle=False, num_workers=0)

print(f"train: {len(train_loader.dataset)}  val: {len(val_loader.dataset)}  test: {len(test_loader.dataset)}")

## Model

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# freeze everything
for param in model.parameters():
    param.requires_grad = False

# swap the head for 4 classes
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

## Training

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(training):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct += (out.argmax(1) == labels).sum().item()

    n = len(loader.dataset)
    return total_loss / n, correct / n

### Phase 1 — train head only (backbone frozen)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

EPOCHS_P1 = 5
best_val_acc = 0.0

for epoch in range(EPOCHS_P1):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader,   criterion)

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), "best_model.pth")

    print(f"epoch {epoch+1}/{EPOCHS_P1}  train_loss={tl:.4f}  val_loss={vl:.4f}  train_acc={ta:.4f}  val_acc={va:.4f}")

### Phase 2 — unfreeze everything and fine-tune

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.fc.parameters(),  "lr": 1e-3},
    {"params": [p for name, p in model.named_parameters() if "fc" not in name], "lr": 1e-4}
])

EPOCHS_P2 = 10

for epoch in range(EPOCHS_P2):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader,   criterion)

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), "best_model.pth")

    print(f"epoch {EPOCHS_P1+epoch+1}/{EPOCHS_P1+EPOCHS_P2}  train_loss={tl:.4f}  val_loss={vl:.4f}  train_acc={ta:.4f}  val_acc={va:.4f}")

print(f"\nbest val acc: {best_val_acc:.4f}")

## Results

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.load_state_dict(torch.load("best_model.pth", map_location=device, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {test_acc:.4f}")

display_labels = [idx_to_class[i] for i in range(len(classes))]
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=display_labels)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {test_acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()